# 06 — Recovery Comparisons
## Trade Winds & Headwinds: USVI, 2000–2025

Indexed comparisons using **2016 = 100** as the pre-hurricane baseline.

Series indexed:
- Total visitor arrivals
- Overnight tourists
- Cruise passengers
- Hotel occupancy rate
- L&H employment (FRED SA annual average)
- Hotel rooms (USVIBER survey)
- A&F real value added (BEA)
- VIIS NP visitation

**Method:** Each series divided by its 2016 value × 100. Series with no 2016 observation are excluded.  
**No single composite score.** Series are compared visually; divergence is the finding.

In [1]:
import sys, importlib
from pathlib import Path
import os

PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import src.cleaning as _c
import src.charts as _ch
import src.hotel_extraction as _he

importlib.reload(_c)
importlib.reload(_ch)
importlib.reload(_he)


# override default notebook template, set background colors to white
# charts.py has a transparent background
import plotly.io as pio

pio.templates["notebook"] = go.layout.Template(
    layout=dict(
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.22,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255,255,255,0)",
            font=dict(size=10.5),
            itemsizing="constant",
            tracegroupgap=0,
        ),
    )
)
pio.templates.default = "plotly_white+notebook"


from src.cleaning import (
    load_usviber_combined,
    load_fred_employment,
    load_bea,
    load_nps_annual,
    load_qcew,
)
from src.hotel_extraction import load_hotel_annual
from src.charts import PALETTE, add_shocks, source_note

pd.set_option("display.float_format", "{:,.1f}".format)
print("Ready.")

Ready.


In [2]:
tour = load_usviber_combined()
fred = load_fred_employment()
bea_real = load_bea("2.3")
viis_ann = load_nps_annual("VIIS")
qcew = load_qcew(deflate=True, cpi_base_year=2016)
qcew_usvi = qcew.query("area_fips == '78000'")
hotel = load_hotel_annual()

fred_ann = (
    fred.groupby("year")["emp_sa"]
    .mean()
    .reset_index()
    .rename(columns={"emp_sa": "emp_sa_avg"})
)
af_real = (
    bea_real[bea_real["label"].str.contains("ccommodation", case=False, na=False)]
    .drop_duplicates(subset="year")[["year", "value"]]
    .rename(columns={"value": "af_real"})
)

# QCEW accommodation employment (NAICS 721)
qcew_usvi_acc = qcew_usvi[qcew_usvi["naics"] == "721"][
    ["year", "annual_avg_emplvl", "avg_annual_pay_real"]
].copy()
qcew_usvi_food = qcew_usvi[qcew_usvi["naics"] == "722"][
    ["year", "annual_avg_emplvl"]
].copy()
qcew_usvi_72 = qcew_usvi[qcew_usvi["naics"] == "72"][
    ["year", "total_annual_wages_real", "avg_annual_pay_real"]
].copy()

print("Data loaded.")
print(
    f"QCEW accommodation employment range: {qcew_usvi_acc.year.min()}-{qcew_usvi_acc.year.max()}"
)
print(f"Hotel nights series range: {hotel.year.min()}-{hotel.year.max()}")

Data loaded.
QCEW accommodation employment range: 2000-2025
Hotel nights series range: 2001-2025


---
## 1. Build the Index Table

In [3]:
BASE_YEAR = 2016


def _index(df, year_col, val_col, base_year=BASE_YEAR):
    """Re-base a count-based series to base_year = 100."""
    base = df.loc[df[year_col] == base_year, val_col]
    if base.empty or pd.isna(base.values[0]) or base.values[0] == 0:
        return None
    base_val = base.values[0]
    out = df[[year_col, val_col]].copy()
    out["idx"] = out[val_col] / base_val * 100
    return out


# Count-based and deflated series eligible for the recovery index
series = {
    # Arrival-mode series
    "Total arrivals": _index(tour, "year", "total_visitors"),
    "Air visitors": _index(tour, "year", "air_visitors"),
    "Cruise passengers": _index(tour, "year", "cruise_passengers"),
    # Hotel / capacity series
    "Hotel rooms": _index(tour, "year", "hotel_rooms"),
    "Nights available": _index(hotel, "year", "nights_available_k"),
    "Nights occupied": _index(hotel, "year", "nights_occupied_k"),
    # Employment series
    "L&H employment (FRED)": _index(fred_ann, "year", "emp_sa_avg"),
    "QCEW accom employment": _index(qcew_usvi_acc, "year", "annual_avg_emplvl"),
    "QCEW food employment": _index(qcew_usvi_food, "year", "annual_avg_emplvl"),
    # Compensation series
    "QCEW real avg pay": _index(qcew_usvi_acc, "year", "avg_annual_pay_real"),
    "QCEW real payroll (72)": _index(qcew_usvi_72, "year", "total_annual_wages_real"),
    # Economic output
    "A&F real value added": _index(af_real, "year", "af_real"),
    # Park visitation
    "VIIS NP visits": _index(
        viis_ann.query("year >= 2000"), "year", "recreation_visitors"
    ),
}

# Visitor-series restriction: overnight tourists excluded; cruise provisional
# (Do not use overnight-tourist series in new comparisons pending source validation)

# Occupancy: shown as percentage-point change, not an index
occ_base = tour.loc[tour["year"] == BASE_YEAR, "occupancy_rate_pct"].values[0]
tour_occ = tour[["year", "occupancy_rate_pct"]].copy()
tour_occ["occ_ppt_vs_2016"] = tour_occ["occupancy_rate_pct"] - occ_base

# Hotel PDF calculated occupancy (nights-based)
hotel_occ_base = hotel.loc[hotel["year"] == BASE_YEAR, "occ_calculated"]
if not hotel_occ_base.empty:
    hotel_occ_ppt_base = hotel_occ_base.values[0]
    hotel["occ_ppt_vs_2016"] = hotel["occ_calculated"] - hotel_occ_ppt_base

# Build combined index table
all_years = sorted(
    set().union(*[set(v["year"]) for v in series.values() if v is not None])
)
idx_wide = pd.DataFrame({"year": all_years})
for name, df in series.items():
    if df is not None:
        idx_wide = idx_wide.merge(
            df[["year", "idx"]].rename(columns={"idx": name}), on="year", how="left"
        )

print(f"Index table: {len(idx_wide)} years x {len(idx_wide.columns)-1} series")
key_years = [2010, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
print(idx_wide[idx_wide["year"].isin(key_years)].set_index("year").round(1).to_string())

Index table: 37 years x 13 series
      Total arrivals  Air visitors  Cruise passengers  Hotel rooms  Nights available  Nights occupied  L&H employment (FRED)  QCEW accom employment  QCEW food employment  QCEW real avg pay  QCEW real payroll (72)  A&F real value added  VIIS NP visits
year                                                                                                                                                                                                                                                      
2010           104.4          84.1              104.6        101.5             104.9            105.4                   94.0                   99.4                  88.2               93.7                    90.8                 104.0           120.0
2015           102.7          95.9              105.7        102.2              99.5            100.1                   96.2                  100.3                  93.7              101.7                    99.5 

In [4]:
# USVIBER tourism employment
usviber_emp = _index(tour, "year", "emp_tourism_total")

if usviber_emp is not None:
    latest = (
        usviber_emp.dropna(subset=["idx"])
        .sort_values("year")
        .iloc[-1]
    )

    baseline = tour.loc[
        tour["year"] == BASE_YEAR, "emp_tourism_total"
    ].iloc[0]

    latest_raw = tour.loc[
        tour["year"] == latest["year"], "emp_tourism_total"
    ].iloc[0]

    print(
        f"USVIBER tourism employment: "
        f"{baseline:,.0f} in {BASE_YEAR}; "
        f"{latest_raw:,.0f} in {int(latest['year'])}; "
        f"{latest['idx']:.1f}% of baseline."
    )

USVIBER tourism employment: 8,430 in 2016; 6,819 in 2022; 80.9% of baseline.


---
## 2. Full Recovery Index Chart

In [5]:
series_styles = [
    # Primary arrival-mode series (air vs cruise)
    ("Air visitors", PALETTE["air_visitors"], "solid", 2.5),
    ("Cruise passengers", PALETTE["cruise"], "solid", 2.5),
    # Secondary series
    ("Total arrivals", PALETTE["total"], "dash", 1.5),
    ("Overnight tourists", PALETTE["tourists"], "dot", 1.5),
    ("Hotel rooms", PALETTE["rooms"], "longdash", 1.5),
    ("L&H employment (FRED)", PALETTE["employment"], "dash", 1.5),
    ("A&F real value added", PALETTE["value_added"], "longdash", 1.5),
    ("VIIS NP visits", PALETTE["nps_viis"], "dashdot", 1.5),
]

fig = go.Figure()
for name, color, dash, width in series_styles:
    if name not in idx_wide.columns:
        continue
    sub = idx_wide[["year", name]].dropna()
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub[name],
            mode="lines",
            name=name,
            line=dict(color=color, dash=dash, width=width),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#ccc", width=1, dash="dot"),
    annotation_text="2016 = 100",
    annotation_position="right",
    annotation_font=dict(size=10, color="#999"),
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )

fig.update_layout(
    title=dict(
        text="USVI Tourism Economy — Recovery Index (2016 = 100)<br>"
        '<span style="font-size:13px;color:#555">Multiple indicators indexed to pre-hurricane baseline. '
        "Divergence indicates uneven recovery across dimensions.</span>",
        font=dict(size=17),
    ),
    yaxis_title="Index (2016 = 100)",
    height=540,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(r=180),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.07,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER, FRED, BEA, NPS IRMA — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 3. Hurricane Shock and Recovery

In [6]:
# Focused view: 2014–2024
focus = idx_wide.query("2014 <= year <= 2025").copy()

fig = go.Figure()
for name, color, dash, width in series_styles:
    if name not in focus.columns:
        continue
    sub = focus[["year", name]].dropna()
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub[name],
            mode="lines+markers",
            name=name,
            line=dict(color=color, dash=dash, width=width),
            marker=dict(size=5),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#ccc", width=1, dash="dot"),
    annotation_text="2016 = 100",
    annotation_position="right",
    annotation_font=dict(size=10, color="#999"),
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )

fig.update_layout(
    title=dict(
        text="Recovery Index — Shock Period and Aftermath (2014–2025)<br>"
        '<span style="font-size:13px;color:#555">2016 = 100. Shows both hurricane and COVID shocks and subsequent recovery.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Index (2016 = 100)",
    height=520,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(r=180),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.07,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER, FRED, BEA, NPS IRMA — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 4. Recovery Scorecard

In [7]:
# Recovery scorecard: index value at key comparison years
# Note: latest available year differs by series (see column labels in bar chart).
comparison_years = [2016, 2017, 2019, 2020, 2022, 2024, 2025]

scorecard = idx_wide[idx_wide["year"].isin(comparison_years)].set_index("year").T
scorecard.columns = [str(y) for y in scorecard.columns]

print("=== Recovery Index Scorecard (2016 = 100) ===")
print("Series with different latest years are labeled in the bar chart below.\n")
print(scorecard.round(1).to_string())

print("\n=== Most recent available value vs. 2016 baseline ===")
# Prefer 2025 > 2024 > 2022 > 2021 in that order
for name in scorecard.index:
    row = scorecard.loc[name]
    latest_col = [
        c
        for c in ["2025", "2024", "2022", "2021"]
        if c in row.index and pd.notna(row[c])
    ]
    if not latest_col:
        continue
    latest_yr = latest_col[0]
    latest = row[latest_yr]
    status = "above 2016" if latest >= 100 else "below 2016"
    print(f"  {name:35s}  latest={latest_yr}: {latest:6.1f}  ({status})")

print()
print("=== Occupancy: percentage-point change from 2016 ===")
print(
    tour_occ[tour_occ["year"].isin(comparison_years)]
    .set_index("year")
    .round(1)
    .to_string()
)

=== Recovery Index Scorecard (2016 = 100) ===
Series with different latest years are labeled in the bar chart below.

                        2016  2017 2019 2020 2022  2024  2025
Total arrivals         100.0  76.0 80.7 30.7 69.4  97.0   NaN
Air visitors           100.0  80.2 80.9 43.6 97.9 109.5   NaN
Cruise passengers      100.0  74.1 80.7 24.8 56.5  90.0   NaN
Hotel rooms            100.0 100.0 79.9 66.9 69.4  81.9   NaN
Nights available       100.0  95.9 55.9 68.9 64.9  79.6  80.1
Nights occupied        100.0  74.5 50.5 26.5 69.7  84.7   NaN
L&H employment (FRED)  100.0  89.9 73.0 63.6 81.5  90.8  84.4
QCEW accom employment  100.0  92.4 49.2 46.2 66.8  84.4  85.1
QCEW food employment   100.0  87.7 97.5 84.0 98.9  99.1  94.9
QCEW real avg pay      100.0 103.1 98.6 96.6 97.3  97.7 100.5
QCEW real payroll (72) 100.0  92.4 66.8 59.6 83.2  94.1  96.2
A&F real value added   100.0  87.9 64.0 46.7  NaN   NaN   NaN
VIIS NP visits         100.0  74.0 32.4 40.7 47.8 102.8 114.5

=== Most rece

In [8]:
# Bar chart: most-recent available index value vs. 2016 baseline
# Note: each series has a different latest available year (shown in label).
latest_vals = {}
for name in idx_wide.columns[1:]:  # skip 'year' column
    for yr in [2025, 2024, 2022, 2021]:
        col_yr = str(yr)
        matches = idx_wide[idx_wide["year"] == yr][name]
        if not matches.empty and pd.notna(matches.values[0]):
            latest_vals[name] = (yr, matches.values[0])
            break

names = list(latest_vals.keys())
labels = [f"{n} ({latest_vals[n][0]})" for n in names]
values = [latest_vals[n][1] for n in names]
bar_colors = [PALETTE["rooms"] if v >= 100 else PALETTE["cruise"] for v in values]

fig = go.Figure(
    go.Bar(
        x=values,
        y=labels,
        orientation="h",
        marker_color=bar_colors,
        text=[f"{v:.0f}" for v in values],
        textposition="outside",
    )
)
fig.add_vline(
    x=100,
    line=dict(color="#444", width=1.5),
    annotation_text="2016 = 100",
    annotation_position="top",
    annotation_font=dict(size=10),
)
fig.update_layout(
    title=dict(
        text="Most Recent Index Value vs. 2016 Baseline<br>"
        '<span style="font-size:13px;color:#555">Blue = at or above 2016; orange = below 2016. '
        "Year in parentheses = latest available year for each series.</span>",
        font=dict(size=17),
    ),
    xaxis_title="Index (2016 = 100)",
    xaxis_range=[0, max(values) * 1.15],
    height=440,
    margin=dict(l=240),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.1,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER, FRED, BEA, NPS IRMA — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

print()
print("=== Occupancy rate (not in index — shown as ppt change from 2016) ===")
occ_key = tour_occ[tour_occ["year"].isin([2016, 2017, 2019, 2020, 2022, 2024])].copy()
print(f"{'Year':6s} {'Rate (%)':10s} {'vs 2016 (ppt)':15s}")
for _, r in occ_key.iterrows():
    print(
        f"  {int(r.year):4d}    {r.occupancy_rate_pct:5.1f}%     {r.occ_ppt_vs_2016:+.1f} pp"
    )


=== Occupancy rate (not in index — shown as ppt change from 2016) ===
Year   Rate (%)   vs 2016 (ppt)  
  2016     56.7%     +0.0 pp
  2017     43.9%     -12.8 pp
  2019     39.7%     -17.0 pp
  2020     37.0%     -19.7 pp
  2022     63.9%     +7.2 pp
  2024     66.3%     +9.6 pp


---
## 5. COVID Shock Depth and Recovery

In [9]:
# Using 2019 as the COVID baseline (last pre-pandemic year)
BASE_COVID = 2019

series_covid = {
    "Total arrivals": _index(tour, "year", "total_visitors", BASE_COVID),
    "Air visitors": _index(tour, "year", "air_visitors", BASE_COVID),
    "Cruise passengers": _index(tour, "year", "cruise_passengers", BASE_COVID),
    "Hotel rooms": _index(tour, "year", "hotel_rooms", BASE_COVID),
    "L&H employment (FRED)": _index(fred_ann, "year", "emp_sa_avg", BASE_COVID),
    "VIIS NP visits": _index(
        viis_ann.query("year >= 2015"), "year", "recreation_visitors", BASE_COVID
    ),
}

fig = go.Figure()
for name, color, dash, width in series_styles:
    if name not in series_covid or series_covid[name] is None:
        continue
    sub = series_covid[name].query("year >= 2017").dropna()
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub["idx"],
            mode="lines+markers",
            name=name,
            line=dict(color=color, dash=dash, width=width),
            marker=dict(size=5),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#ccc", width=1, dash="dot"),
    annotation_text="2019 = 100",
    annotation_position="right",
    annotation_font=dict(size=10, color="#999"),
)
fig.add_vline(
    x=2020,
    line=dict(color="#555", width=1.5, dash="dash"),
    annotation_text="COVID-19",
    annotation_position="top",
    annotation_font=dict(size=10, color="#444"),
)

fig.update_layout(
    title=dict(
        text="COVID Shock and Recovery (2019 = 100)<br>"
        '<span style="font-size:13px;color:#555">Indexed to last pre-pandemic year. Note: hotel room count already post-hurricane in 2019.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Index (2019 = 100)",
    height=500,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(r=180),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.07,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER, FRED, NPS IRMA — author calculations</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
fig.show()

---
## 7. Extended Recovery: Employment, Compensation, and Utilization

New series added from QCEW and hotel PDF data sources:

| Series | Source | Coverage | Notes |
|---|---|---|---|
| QCEW accommodation employment | BLS QCEW | 2000–2025 | Covered jobs, NAICS 721 |
| QCEW food services employment | BLS QCEW | 2000–2025 | Covered jobs, NAICS 722 |
| QCEW real avg pay | BLS QCEW + CPI-U | 2000–2025 | NAICS 721, approx 2016 $ |
| QCEW real total payroll | BLS QCEW + CPI-U | 2000–2025 | NAICS 72, approx 2016 $ |
| Hotel nights available | USVIBER PDF | 2006–2024 | Thousands, full-year only |
| Hotel nights occupied | USVIBER PDF | 2000–2024 | Thousands, full-year only |

**Visitor-series restriction:** Overnight tourist arrivals are excluded from recovery comparisons in this section pending source validation of the USVIBER methodology. Cruise passenger comparisons are retained provisionally.


In [10]:
# --- QCEW and hotel recovery comparisons (2016 = 100) ---
QCEW_HOTEL_SERIES = [
    ("Nights available", PALETTE["rooms"], "solid", 2.2, 1.0),
    ("Nights occupied", PALETTE["employment"], "dash", 2.2, 1.0),
    ("L&H employment (FRED)", PALETTE["total"], "dot", 1.6, 0.70),
    ("QCEW accom employment", PALETTE["tourists"], "longdash", 1.6, 0.70),
    ("QCEW food employment", PALETTE["cruise"], "longdash", 1.6, 0.70),
    ("QCEW real avg pay", PALETTE["air_visitors"], "dashdot", 1.6, 0.70),
    ("A&F real value added", PALETTE["value_added"], "dash", 1.4, 0.55),
]

fig = go.Figure()
for name, color, dash, lw, op in QCEW_HOTEL_SERIES:
    if name not in idx_wide.columns:
        continue
    sub = idx_wide[idx_wide["year"].between(2000, 2024)][["year", name]].dropna()
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub[name],
            mode="lines",
            name=name,
            opacity=op,
            line=dict(color=color, dash=dash, width=lw),
        )
    )

fig.add_hline(
    y=100,
    line=dict(color="#d1d5db", width=1, dash="dot"),
    annotation_text="2016 = 100",
    annotation_position="right",
    annotation_font=dict(size=9, color="#aaa"),
)
fig.update_layout(
    title=dict(
        text="<b>Recovery Index: Hotel Utilization, Employment, Compensation (2016 = 100)</b>",
        font=dict(size=16),
        x=0.0,
    ),
    yaxis_title="Index (2016 = 100)",
    height=480,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.35,
        xanchor="center",
        x=0.5,
        font=dict(size=10),
    ),
    margin=dict(l=60, r=24, t=52, b=120),
    annotations=[
        source_note(
            "BLS QCEW, USVIBER Hotel PDFs, FRED, BEA. Real values: CPI-U proxy, 2016 base."
        )
    ],
)
add_shocks(fig)
fig.show()

In [11]:
# --- Occupancy: ppt change from 2016 (rate-based, not index) ---
fig = go.Figure()

# USVIBER annual occupancy (ppt)
fig.add_trace(
    go.Scatter(
        x=tour_occ["year"],
        y=tour_occ["occ_ppt_vs_2016"],
        mode="lines+markers",
        name="Occupancy ppt change (USVIBER annual)",
        line=dict(color=PALETTE["air_visitors"], width=2.4),
        marker=dict(size=5),
    )
)
# Hotel PDF calculated occupancy (ppt)
if "occ_ppt_vs_2016" in hotel.columns:
    fig.add_trace(
        go.Scatter(
            x=hotel["year"],
            y=hotel["occ_ppt_vs_2016"],
            mode="lines+markers",
            name="Occupancy ppt change (PDF nights-based)",
            line=dict(color=PALETTE["total"], width=1.8, dash="dash"),
            marker=dict(size=4),
            opacity=0.75,
        )
    )

fig.add_hline(
    y=0,
    line=dict(color="#d1d5db", width=1, dash="dot"),
    annotation_text="2016 baseline",
    annotation_position="right",
    annotation_font=dict(size=9, color="#aaa"),
)
fig.update_layout(
    title=dict(
        text="<b>Hotel Occupancy Rate: Percentage-Point Change vs. 2016</b>",
        font=dict(size=16),
        x=0.0,
    ),
    yaxis_title="Percentage-point change from 2016 rate",
    height=400,
    annotations=[
        source_note(
            "USVIBER Annual Tourism Indicator 2024; USVIBER Hotel PDFs (nights-based)."
        )
    ],
)
add_shocks(fig)
fig.show()

print("Occupancy ppt change from 2016 baseline (56.7%):")
occ_compare = tour_occ[["year", "occupancy_rate_pct", "occ_ppt_vs_2016"]].copy()
occ_compare.columns = ["year", "occupancy_%", "vs_2016_ppt"]
print(occ_compare[occ_compare.year.between(2014, 2024)].set_index("year").to_string())

Occupancy ppt change from 2016 baseline (56.7%):
      occupancy_%  vs_2016_ppt
year                          
2014         53.8         -2.9
2015         57.1          0.4
2016         56.7          0.0
2017         43.9        -12.8
2018         31.3        -25.4
2019         39.7        -17.0
2020         37.0        -19.7
2021         62.4          5.7
2022         63.9          7.2
2023         58.7          2.0
2024         66.3          9.6


---
## 6. Preliminary Observations

*(These are analytical observations from the data, not conclusions for publication. They inform what deserves deeper investigation.)*

Run the scorecard cell and examine the index table before forming any conclusions. Key questions to explore in the Field Note phase:

1. **Did total visitor arrivals recover to 2016 levels by 2024?** Check the scorecard.
2. **Did overnight tourists recover at the same rate as cruise passengers?** Look for divergence in the composition.
3. **Did employment recover in step with visitor volume?** The Census 2022 accommodation employment (1,650 jobs) vs. 2012 (3,876) suggests structural change beyond what visit counts show.
4. **Did room inventory recover?** The room count trajectory (USVIBER vs. Census) tells different stories — investigate which is more reliable.
5. **Is 2024 occupancy rate (66.3%) telling a different story than absolute visitor counts?** High occupancy on a smaller room base may represent profitable operation at reduced scale, not full recovery.
6. **Why is VIIS NP visitation so different from territory-wide arrivals?** The park draws overnight tourists, not cruise passengers — the VIIS trajectory should track overnight tourist arrivals more closely than total arrivals.

---
## 8. Island-Level Recovery: STT vs. STX

The territory-wide recovery aggregates mask meaningful differences between the two island groups. Island-level arrivals and hotel statistics are compared on a 2016 = 100 basis.

Note: island-level hotel data begin 2007; the full 2000-2006 period is not available at island level.


In [12]:
from src.cleaning import load_usviber_island_arrivals, load_hotel_island
island_arr = load_usviber_island_arrivals()
hotel_island = load_hotel_island()

# Index all island series to 2016 = 100
BASE = 2016
idx = island_arr.copy()
for col in ['stt_air', 'stt_cruise', 'stx_air', 'stx_cruise']:
    base_v = idx.loc[idx['year']==BASE, col].values
    if len(base_v) and not pd.isna(base_v[0]) and base_v[0] > 0:
        idx[col+'_idx'] = idx[col] / base_v[0] * 100

b16h = hotel_island[hotel_island['year']==BASE].iloc[0]
hotel_island['stt_avail_idx'] = hotel_island['stt_available'] / b16h['stt_available'] * 100
hotel_island['stx_avail_idx'] = hotel_island['stx_available'] / b16h['stx_available'] * 100
hotel_island['stt_occ_idx']   = hotel_island['stt_occupied']  / b16h['stt_occupied']  * 100
hotel_island['stx_occ_idx']   = hotel_island['stx_occupied']  / b16h['stx_occupied']  * 100

arr_hotel = idx[['year','stt_air_idx','stx_air_idx','stt_cruise_idx','stx_cruise_idx']].merge(
    hotel_island[['year','stt_avail_idx','stx_avail_idx','stt_occ_idx','stx_occ_idx']],
    on='year', how='outer'
)

print('Island-level recovery index 2024 (all series vs 2016=100):')
row = arr_hotel[arr_hotel['year']==2024].iloc[0]
print(row.to_string())


Island-level recovery index 2024 (all series vs 2016=100):
year             2,024.0
stt_air_idx        110.2
stx_air_idx        106.4
stt_cruise_idx      90.4
stx_cruise_idx     144.5
stt_avail_idx       74.9
stx_avail_idx       94.8
stt_occ_idx         80.7
stx_occ_idx        101.2


In [13]:
# STT panel: air arrivals, cruise arrivals, available room-nights, occupied room-nights
fig = go.Figure()
for col, name, color, dash in [
    ('stt_air_idx',    'Air arrivals',          PALETTE['air_visitors'], 'solid'),
    ('stt_cruise_idx', 'Cruise arrivals',        PALETTE['cruise'],       'solid'),
    ('stt_avail_idx',  'Room-nights available',  PALETTE['rooms'],        'dot'),
    ('stt_occ_idx',    'Room-nights occupied',   PALETTE['employment'],   'dash'),
]:
    sub = arr_hotel[['year', col]].dropna()
    fig.add_trace(go.Scatter(x=sub['year'], y=sub[col], mode='lines', name=name,
                             line=dict(color=color, width=2.0, dash=dash)))
fig.add_hline(y=100, line=dict(color='#aaa', width=1, dash='dot'),
              annotation_text='2016 = 100', annotation_position='right',
              annotation_font=dict(size=9, color='#999'))
fig.update_layout(title='St. Thomas-St. John: Arrivals and Hotel Capacity (2016 = 100)',
    yaxis=dict(title='Index (2016 = 100)', range=[0, 130]),
    xaxis=dict(range=[2007, 2025]), height=450,
    annotations=[_ch.source_note('USVIBER island-level data')])
_ch.add_shocks(fig)
fig.show()


In [14]:
# STX panel
fig = go.Figure()
for col, name, color, dash in [
    ('stx_air_idx',    'Air arrivals',          PALETTE['air_visitors'], 'solid'),
    ('stx_cruise_idx', 'Cruise arrivals',        PALETTE['cruise'],       'solid'),
    ('stx_avail_idx',  'Room-nights available',  PALETTE['rooms'],        'dot'),
    ('stx_occ_idx',    'Room-nights occupied',   PALETTE['employment'],   'dash'),
]:
    sub = arr_hotel[['year', col]].dropna()
    fig.add_trace(go.Scatter(x=sub['year'], y=sub[col], mode='lines', name=name,
                             line=dict(color=color, width=2.0, dash=dash)))
fig.add_hline(y=100, line=dict(color='#aaa', width=1, dash='dot'),
              annotation_text='2016 = 100', annotation_position='right',
              annotation_font=dict(size=9, color='#999'))
fig.update_layout(title='St. Croix: Arrivals and Hotel Capacity (2016 = 100)',
    yaxis=dict(title='Index (2016 = 100)', range=[0, 170]),
    xaxis=dict(range=[2007, 2025]), height=450,
    annotations=[_ch.source_note('USVIBER island-level data; STX cruise has historical low 2003-2008')])
_ch.add_shocks(fig)
fig.show()


### Island-Level Recovery Observations

- **STT** air arrivals (110% of 2016) exceeded the territory average; cruise (90%) matched it. Available room-nights remain at 75%, occupied room-nights at 81%.
- **STX** air arrivals (106%) were also above 2016. Cruise traffic (144%) greatly exceeded 2016, partly reflecting that STX cruise was historically depressed during 2003-2008. Available room-nights (95%) and occupied room-nights (101%) are both near or above 2016.
- The territory-wide accommodation shortfall (82% of 2016 hotel rooms) is therefore primarily an STT story. St. Croix hotel capacity has largely recovered.
- STX cruise growth should be interpreted with caution: the 2016 STX cruise baseline (117K) was itself well below the early-2000s peaks (230K+).
